# 06 · 回测过拟合：试得越多，越容易“发现”假策略

**学习目标**：
1. 亲眼看到**选择偏差**：200 个毫无预测能力的策略里，总能挑出一个回测很漂亮的；
2. 学会估算“试了 N 次之后，最好的结果本来就该有多好”；
3. 用**折减夏普比率（Deflated Sharpe Ratio, DSR）**给挑出来的结果打折，判断它是不是真本事。

**数据**：200 个策略、5 年（1250 个交易日）的**纯随机**日收益（固定随机种子）。每个策略的真实夏普比率都是 0，所以任何“好结果”都只是运气。

**承接**：[02 号练习](02_factor_backtest_lookahead.ipynb)讲的是前视偏差（偷看未来）；本练习讲的是另一种更隐蔽的错误：**没偷看未来，但试了太多次**。方法来自 Bailey & López de Prado（2014）的论文，公式实现在 [`scripts/backtest_stats.py`](../scripts/backtest_stats.py)。

> 本仓库仅供学习，所有内容不构成投资建议。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 复用 scripts/ 里的公式实现
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import backtest_stats as bs

pd.set_option("display.float_format", "{:.4f}".format)
PERIODS = 252


def ann_sharpe(r: pd.DataFrame | pd.Series):
    """年化夏普比率（无风险利率按 0）。"""
    return r.mean() / r.std() * np.sqrt(PERIODS)

## 1. 生成 200 个“没有本事”的策略

每个策略的日收益独立同分布：均值 0、日波动 1%。前 3 年（750 天）当作**样本内**，用来挑策略；后 2 年（500 天）当作**样本外**，检验挑出来的策略。

In [ ]:
rng = np.random.default_rng(2026)
n_strats, n_days, n_is = 200, 1250, 750
dates = pd.bdate_range("2021-01-01", periods=n_days)
rets = pd.DataFrame(rng.normal(0.0, 0.01, (n_days, n_strats)), index=dates,
                    columns=[f"S{i:03d}" for i in range(n_strats)])
rets_is, rets_oos = rets.iloc[:n_is], rets.iloc[n_is:]

sr_is, sr_oos = ann_sharpe(rets_is), ann_sharpe(rets_oos)
pd.DataFrame({"in-sample": sr_is.describe(), "out-of-sample": sr_oos.describe()})

## 2. 挑出样本内最好的那个

这就是很多“回测研究”的真实过程：换参数、换股票池、换调仓频率，最后只汇报最好的一个。

In [ ]:
best = sr_is.idxmax()
print(f"样本内最好的策略 {best}：年化夏普 {sr_is[best]:.2f}")
print(f"同一个策略在样本外：年化夏普 {sr_oos[best]:.2f}")
print(f"样本内夏普排名前 10 的策略，样本外平均夏普：{sr_oos[sr_is.nlargest(10).index].mean():.2f}")

nav = (1 + rets[best]).cumprod()
ax = nav.plot(figsize=(9, 4), title=f"Best in-sample strategy {best} (pure noise)")
ax.axvline(rets.index[n_is], color="gray", linestyle="--")
ax.text(rets.index[n_is], nav.max(), "  out-of-sample →", va="top")
ax.set_ylabel("NAV")
plt.show()

## 3. 这个最高值，其实是可以预期的

在“所有策略都没本事”的假设下，夏普比率的估计值本身有抽样误差。试了 N 次之后，**最大值的期望**可以用论文的公式算出来（只需要 N 和各次试验夏普比率的方差）。实际挑出来的最高值会在这个期望值上下波动。

In [ ]:
sr_is_daily = rets_is.mean() / rets_is.std()           # 单期（日度）口径，公式要求不年化
sr0 = bs.expected_max_sharpe(n_strats, sr_is_daily.var())

print(f"公式预测的“200 次里最好的那个”：年化 {sr0 * np.sqrt(PERIODS):.2f}")
print(f"实际挑出来的最高值：            年化 {sr_is[best]:.2f}")
print(f"理论上单次估计的标准误：        年化 {np.sqrt(PERIODS / n_is):.2f}（约等于 √(年交易日/样本天数)）")

In [ ]:
ns = np.unique(np.logspace(0, 3, 40).astype(int))
thresholds = [bs.expected_max_sharpe(n, 1 / n_is) * np.sqrt(PERIODS) for n in ns]
ax = pd.Series(thresholds, index=ns).plot(logx=True, figsize=(9, 4),
                                          title="Expected best Sharpe from N skill-less trials (3-year sample)")
ax.set_xlabel("number of trials N (log scale)")
ax.set_ylabel("annualized Sharpe")
plt.show()

## 4. 用 DSR 打折

- **PSR**（概率夏普比率）：在只做了**一次**试验的前提下，真实夏普大于 0 的概率。
- **DSR**：把门槛从 0 提高到“N 次试验的期望最大值”之后的 PSR。习惯上 **DSR ≥ 0.95** 才算显著。

In [ ]:
r = rets_is[best]
sk, ku = bs.skew_kurt(r.tolist())
sr_best = sr_is_daily[best]
summary = pd.Series({
    "annualized Sharpe (in-sample)": sr_is[best],
    "PSR, pretending it was the only trial": bs.probabilistic_sharpe_ratio(sr_best, 0.0, n_is, sk, ku),
    f"DSR, honest about {n_strats} trials": bs.deflated_sharpe_ratio(sr_best, n_strats, sr_is_daily.var(), n_is, sk, ku),
})
summary

## 5. 怎么读这个结果

- **样本内最好的策略**夏普比率看起来不错，但它只是 200 次抽签里最幸运的一次。到了样本外，它落回所有策略的正常波动范围（2 年样本的夏普比率本身就有约 ±0.7 的抽样误差），样本内前 10 名在样本外的平均夏普也接近 0。
- 如果**隐瞒试验次数**，PSR 会告诉你“几乎可以肯定有真本事”；如实写上试了 200 次，DSR 就会显示它和运气没有区别。
- 所以评估任何回测（包括 AI 智能体自动生成的回测），第一个问题是：**一共试了多少个版本？** 第二个问题是：**样本外表现如何？**
- 论文自己的数值例子（年化夏普 2.5、5 年日度数据、试了 100 次）同样不显著，运行 `python scripts/backtest_stats.py` 可以复现。

## 练习

1. 把 `n_strats` 改成 20 和 2000，样本内最高夏普和 DSR 分别怎么变？
2. 给其中一个策略加上真实收益（例如每天多 0.05%，年化夏普约 0.8），DSR 能把它挑出来吗？把样本加长到 10 年再试一次。
3. 回到 02 号练习，把动量的回看期从 1 个月试到 12 个月，挑出最好的一个，再用“试了 12 次”算 DSR。
4. 读 Harvey、Liu、Zhu 的论文（新因子的 t 值应超过 3.0，而不是 2.0），想一想它和 DSR 解决的是不是同一个问题。